In [5]:
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [4]:
df_1 = pd.read_parquet("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/code/01_ETL/Bronze/00_Manja_test/output/parquet/df_1_train_cleaned.parquet")
print(df_1.shape)
df_1.head()

(83948, 74)


,flight_date,movement_date,flight_number,airline,airport_origin,airport_destination,terminal_departure,terminal_arrival,scheduled_departure,scheduled_arrival,...,GREVE_ORLY,nombre_departs_source,nombre_arrivees_source,somme_depart_arrivee_source,congestion_source,nombre_departs_destination,nombre_arrivees_destination,somme_depart_arrivee_destination,congestion_destination,retard arrivée
0,2025-10-11,2025-10-11,5O 701,ASL Airlines France,CDG,MRS,None,None,2025-10-11 01:39+02:00,2025-10-11 02:34+02:00,...,Non,616.0,620.0,1236.0,0.0,119.0,113.0,232.0,1.0,0
1,2025-10-11,2025-10-11,AF 6001,Air France,MRS,ORY,1,2,2025-10-11 06:20+02:00,2025-10-11 07:45+02:00,...,Non,119.0,113.0,232.0,1.0,299.0,299.0,598.0,0.0,0
2,2025-10-11,2025-10-11,AF 6004,Air France,ORY,MRS,3,1,2025-10-11 09:00+02:00,2025-10-11 10:20+02:00,...,Non,299.0,299.0,598.0,0.0,119.0,113.0,232.0,1.0,0
3,2025-10-11,2025-10-11,AF 6009,Air France,MRS,ORY,1,2,2025-10-11 15:40+02:00,2025-10-11 17:05+02:00,...,Non,119.0,113.0,232.0,1.0,299.0,299.0,598.0,0.0,0
4,2025-10-11,2025-10-11,AF 6101,Air France,TLS,ORY,None,2,2025-10-11 06:30+02:00,2025-10-11 07:55+02:00,...,Non,78.0,73.0,151.0,0.0,299.0,299.0,598.0,0.0,0


In [7]:
df_1_train_reg = df_1[df_1["arrival_delay_min"].notna()].copy()

y_reg = df_1_train_reg["arrival_delay_min"]

X_reg = df_1_train_reg.drop(columns=[
    "retard arrivée",
    "arrival_delay_min",
    "status"
], errors="ignore")

datetime_cols = [
    "flight_date",
    "movement_date",
    "scheduled_departure",
    "scheduled_arrival",
    "time_dep",
    "time_arr"
]

def add_datetime_features_safe(df, datetime_cols):
    df = df.copy()
    bad_datetime_cols = []

    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

            if not is_datetime64_any_dtype(df[col]):
                bad_datetime_cols.append(col)

    usable_datetime_cols = [
        col for col in datetime_cols
        if col in df.columns and col not in bad_datetime_cols
    ]

    if "flight_date" in usable_datetime_cols:
        df["flight_month"] = df["flight_date"].dt.month
        df["flight_day"] = df["flight_date"].dt.day
        df["flight_dayofweek"] = df["flight_date"].dt.dayofweek

    if "scheduled_departure" in usable_datetime_cols:
        df["sched_dep_hour"] = df["scheduled_departure"].dt.hour
        df["sched_dep_minute"] = df["scheduled_departure"].dt.minute

    if "scheduled_arrival" in usable_datetime_cols:
        df["sched_arr_hour"] = df["scheduled_arrival"].dt.hour
        df["sched_arr_minute"] = df["scheduled_arrival"].dt.minute

    print("Colonnes datetime problématiques droppées :", bad_datetime_cols)

    df = df.drop(columns=datetime_cols, errors="ignore")

    return df

X_reg = add_datetime_features_safe(X_reg, datetime_cols)

valid_idx = y_reg.notna()
X_reg = X_reg.loc[valid_idx].copy()
y_reg = y_reg.loc[valid_idx].copy()

num_cols_reg = X_reg.select_dtypes(include=["number"]).columns.tolist()
cat_cols_reg = X_reg.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

num_pipeline_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

cat_pipeline_reg = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_reg = ColumnTransformer([
    ("num", num_pipeline_reg, num_cols_reg),
    ("cat", cat_pipeline_reg, cat_cols_reg)
])

model_reg = Pipeline([
    ("preprocessor", preprocessor_reg),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2,
        min_samples_split=5,
        max_depth=None
    ))
])

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

model_reg.fit(X_train_reg, y_train_reg)

y_pred_reg = model_reg.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = root_mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("DF_1 REGRESSION RESULTS")
print("MAE :", mae)
print("RMSE :", rmse)
print("R2 :", r2)

/var/folders/p7/7w66qcfn4yj82k3sbs5fp74h0000gn/T/ipykernel_8307/3936386003.py:26: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df[col] = pd.to_datetime(df[col], errors="coerce")
/var/folders/p7/7w66qcfn4yj82k3sbs5fp74h0000gn/T/ipykernel_8307/3936386003.py:26: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df[col] = pd.to_datetime(df[col], errors="coerce")


Colonnes datetime problématiques droppées : ['scheduled_departure', 'scheduled_arrival']
DF_1 REGRESSION RESULTS
MAE : 2.258527734699852
RMSE : 8.010568358465282
R2 : 0.9089017223622331


In [9]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from mlflow.models import infer_signature
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

from dotenv import load_dotenv
load_dotenv()


MLFLOW_TRACKING_URI = "https://stoneray-ppml-mlflow.hf.space"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("FlyOnTime_regressor")

# sécurité
while mlflow.active_run():
    mlflow.end_run()

# prédictions
y_pred_reg = model_reg.predict(X_test_reg)

# métriques
mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = root_mean_squared_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

# signature
signature = infer_signature(X_test_reg, y_pred_reg)

with mlflow.start_run(run_name="RandomForest_Regressor") as run:
    # params
    mlflow.log_params({
        "model_type": "RandomForestRegressor",
        "target": "arrival_delay_min",
        "n_estimators": 300,
        "random_state": 42,
        "n_jobs": -1,
        "min_samples_leaf": 2,
        "min_samples_split": 5,
        "max_depth": None,
        "test_size": 0.2,
        "n_features_train": X_train_reg.shape[1],
        "n_rows_train": X_train_reg.shape[0],
        "n_rows_test": X_test_reg.shape[0],
    })

    # metrics
    mlflow.log_metric("mae", float(mae))
    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("r2", float(r2))

    # modèle complet pipeline = preprocessing + modèle
    mlflow.sklearn.log_model(
        sk_model=model_reg,
        name="RandomForest_Regressor_Manjaka",
        registered_model_name="RandomForest_Regressor_registered",
        signature=signature,
        input_example=X_test_reg.head(5)
    )

    print("Run ID :", run.info.run_id)
    print("MAE    :", mae)
    print("RMSE   :", rmse)
    print("R2     :", r2)

/Users/manjakaranjatoson/anaconda3/envs/env_ppml/lib/python3.11/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/18 09:49:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, whic

Run ID : 9e59e6bb49324d9fac9da1c29eada452
MAE    : 2.2585277346998516
RMSE   : 8.010568358465282
R2     : 0.9089017223622331
🏃 View run RandomForest_Regressor at: https://stoneray-ppml-mlflow.hf.space/#/experiments/10/runs/9e59e6bb49324d9fac9da1c29eada452
🧪 View experiment at: https://stoneray-ppml-mlflow.hf.space/#/experiments/10
